🏗️ 02 — Refactoring Demo: MEGA-Script → 3-Module Tower!

**Topic 14.02 | Production_Ready_Mindset — Part 2 of 4 (demo notebook)**

🌍 **Hook:** A sensor-ETL script — a 100-line THEPLA: reading, cleaning,
avg-spotting, report-printing all in one file. Problem: the cramped work — you go to
fix the parser and the REPORT breaks! 😱 From today on it's live: the MEGA's demolition,
3 small rooms (load → compute → report) — let the tower stand! 🏗️

## 📜 BEFORE — MEGA `sensor_etl.py` (concept-print!)

```python
# 1-file-etl (meiodule-style ganda):
raw = open("sensors.csv").readlines()
rows = []
for l in raw:
    if l.startswith("#") or not l.strip(): continue
    p = l.split(",")
    if p[2].strip() == "OK":
        rows.append((p[0], float(p[1])))
tot = 0
for _, v in rows: tot += v
avg = round(tot / len(rows), 1)
print(f"rows={len(rows)} avg={avg}")
best = rows[0]
for r in rows:
    if r[1] > best[1]: best = r
print("top:", best[0], best[1])
# + report lines CSV likhna + email-format — SAB YAHIN! 😱
```
Anyone touching this = EVERYONE confused. We'll split it into `etl_load` / `etl_compute` / `etl_report`!

## 🐣 Step 1 — Demo-CSV Seed 🧺 (these are the supplies!)

In [1]:
from pathlib import Path

Path("sensors_demo.csv").write_text(
    "# log-start\ns1,42.5,OK\ns2,17.3,ERR\ns3,60.0,OK\ns4,55.2,OK\ns5,8.0,OK\n")
print(Path("sensors_demo.csv").read_text())
# Expected OUTPUT:
# # log-start
# s1,42.5,OK
# s2,17.3,ERR
# s3,60.0,OK
# s4,55.2,OK
# s5,8.0,OK

# log-start
s1,42.5,OK
s2,17.3,ERR
s3,60.0,OK
s4,55.2,OK
s5,8.0,OK



## 🐥 Step 2 — Write the 3 Modules (each-room-expert!) 🧩

In [2]:
# ROOM 1 — etl_load.py: sirf file-parsing (csv→rows!)
Path("etl_load.py").write_text('''
"""etl_load — CSV-file ko saaf rows deta room 📥"""

def load_ok_rows(fp: str) -> list[tuple[str, float]]:
    rows = []
    for line in open(fp):
        if line.startswith("#") or not line.strip():
            continue
        name, val, status = (x.strip() for x in line.split(","))
        if status == "OK":
            rows.append((name, float(val)))
    return rows

if __name__ == "__main__":
    print("SELF:", load_ok_rows("sensors_demo.csv"))
''')

# ROOM 2 — etl_compute.py: sirf maths (avg/top!)
Path("etl_compute.py").write_text('''
"""etl_compute — rows ko stats 🧮"""

def rows_avg(rows) -> float:
    return round(sum(v for _, v in rows) / len(rows), 1) if rows else 0.0

def rows_top(rows):
    return max(rows, key=lambda r: r[1]) if rows else None

if __name__ == "__main__":
    print("SELF:", rows_avg([("a", 10.0), ("b", 20.0)]))
''')

# ROOM 3 — etl_report.py: sirf presentation (report-lines!)
Path("etl_report.py").write_text('''
"""etl_report — stats ko baashah-letter-roop 🖨️"""

def make_report(rows, avg: float, top) -> str:
    return (f"SENSOR-CHAAT\\n"
            f"rows={len(rows)} avg={avg}\\n"
            f"champion={top[0]} @ {top[1]}\\n"
            f"credentials: load→compute→report tower! 🏗️")

if __name__ == "__main__":
    print(make_report([("sx", 1.0)], 1.0, ("sx", 1.0)))
''')

print("3 rooms likh diye ✓ (etl_load📥 / etl_compute🧮 / etl_report🖨️)")
# Expected OUTPUT:
# 3 rooms likh diye ✓ (etl_load📥 / etl_compute🧮 / etl_report🖨️)

3 rooms likh diye ✓ (etl_load📥 / etl_compute🧮 / etl_report🖨️)


## 🐔 Step 3 — MAIN Runner: Tower Built! 🏗️ (import → chain → cleanup)

In [3]:
import sys, importlib, shutil
sys.path.insert(0, ".")

importlib.invalidate_caches()   # fresh-written files ETIC-recognize ete!
etl_load    = importlib.import_module("etl_load")
etl_compute = importlib.import_module("etl_compute")
etl_report  = importlib.import_module("etl_report")

rows = etl_load.load_ok_rows("sensors_demo.csv")
avg  = etl_compute.rows_avg(rows)
top  = etl_compute.rows_top(rows)
report = etl_report.make_report(rows, avg, top)
print(report)

assert rows == [("s1", 42.5), ("s3", 60.0), ("s4", 55.2), ("s5", 8.0)], "4 OK rows!"
assert avg == 41.4 and top == ("s3", 60.0), "avg 41.43→41.4, top=s3@60!"

# cleanup — demo-artifacts asali-repo me rakhte, NB me saaf!
for f in ("etl_load.py", "etl_compute.py", "etl_report.py", "sensors_demo.csv"):
    Path(f).unlink()
shutil.rmtree("__pycache__", ignore_errors=True)
print("tower tour ✓ — files saaf (asali repo me YEH structure permanents & survives!) 🏗️")
# Expected OUTPUT:
# SENSOR-CHAAT
# rows=4 avg=41.4
# champion=s3 @ 60.0
# credentials: load→compute→report tower! 🏗️
# tower tour ✓ — files saaf (asali repo me YEH structure permanents & survives!) 🏗️

SENSOR-CHAAT
rows=4 avg=41.4
champion=s3 @ 60.0
credentials: load→compute→report tower! 🏗️
tower tour ✓ — files saaf (asali repo me YEH structure permanents & survives!) 🏗️


---
## ⚠️ 3 Refactor-Specific Mistakes
| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | DIRECT-run logic lying open in module files | every file: a `__main__`-guard 🛡️ |
| 2 | chain = import-cycle (load→report→load!) 🔒 | one-way flow (load→compute→report!) |
| 3 | skipping tests during the refactor (no same-output proof!) | asserts on top/avg EXACT |

## 📌 Recap — 15 Seconds
- The 3-room pattern: 📥 load (files) → 🧮 compute (pure!) → 🖨️ report (display)
- the reward of modularity: fix the parser, and the report doesn't break! 💪

**Next topic:** `03_Logging_Monitoring_Mindset` — the "what to log, WHERE to
log, HOW to drink it" mindset (the BLACK-BOX flight-recorder of production!) 🛩️